In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
import joblib

# === STEP 1: Load Data ===
df_old = pd.read_csv("Cleaned_Merged_data_alarm.csv")
df_new = pd.read_csv("DataAlarm_25Januari_20April.csv")

# === STEP 2: Preprocessing kolom waktu ===
for df in [df_old, df_new]:
    df["Last Occurred (ST)"] = pd.to_datetime(df["Last Occurred (ST)"], errors="coerce").astype(int) // 10**9
    df["Acknowledged On (ST)"] = pd.to_datetime(df["Acknowledged On (ST)"], errors="coerce").astype(int) // 10**9

# Gabungkan data
df = pd.concat([df_old, df_new], ignore_index=True).drop_duplicates()

# === STEP 3: Fitur dan Target ===
X = df.drop(columns=["Severity"])
y = df["Severity"]

# === STEP 4: Normalisasi ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, "scaler_lstm.pkl")

# === STEP 5: Split + Balancing ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# === STEP 6: Reshape for LSTM ===
X_train_lstm = X_train_bal.reshape((X_train_bal.shape[0], 1, X_train_bal.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# === STEP 7: Build Model ===
model = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(1, X_train_lstm.shape[2])),
    Dropout(0.3),
    BatchNormalization(),
    LSTM(32, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')  # 4 kelas Severity
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# === STEP 8: Train Model ===
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train_lstm, y_train_bal, validation_data=(X_test_lstm, y_test),
                    epochs=50, batch_size=32, callbacks=[early_stop])

# === STEP 9: Evaluasi ===
y_pred = np.argmax(model.predict(X_test_lstm), axis=1)

print("\n=== Evaluasi Model Retrain LSTM ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# === STEP 10: Simpan Model ===
model.save("retrained_lstm_alarm_model.h5")
print("✅ Model retrain disimpan ke retrained_lstm_alarm_model.h5")


Epoch 1/50


/home/ubuntu/ryh_training/venv/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4790/4790 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.7688 - loss: 0.5766 - val_accuracy: 0.8687 - val_loss: 0.3494
Epoch 2/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8668 - loss: 0.3562 - val_accuracy: 0.8931 - val_loss: 0.2933
Epoch 3/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8755 - loss: 0.3312 - val_accuracy: 0.8944 - val_loss: 0.3019
Epoch 4/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8814 - loss: 0.3167 - val_accuracy: 0.9033 - val_loss: 0.2935
Epoch 5/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8845 - loss: 0.3110 - val_accuracy: 0.9031 - val_loss: 0.2797
Epoch 6/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8896 - loss: 0.2969 - val_accuracy: 0.8910 - val_loss: 0.3034
Epoch 7/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8916 - loss: 0.2944 - val_accuracy: 0.9019 - val_loss: 0.2888
Epoch 8/50
4790/4790 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8942 - loss: 0.2879 - val


=== Evaluasi Model Retrain LSTM ===
Accuracy : 0.9109449699928797
Confusion Matrix:
 [[8383   55 1012  130]
 [  97  994   41   40]
 [  59   26 2241    0]
 [ 157  129    5 6293]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.88      0.92      9580
           1       0.83      0.85      0.84      1172
           2       0.68      0.96      0.80      2326
           3       0.97      0.96      0.96      6584

    accuracy                           0.91     19662
   macro avg       0.86      0.91      0.88     19662
weighted avg       0.93      0.91      0.91     19662

✅ Model retrain disimpan ke retrained_lstm_alarm_model.h5


In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

# === Fungsi untuk proses input CSV dari teknisi ===
def process_uploaded_alarm(csv_path: str):
    df = pd.read_csv(csv_path)
    df["Last Occurred (ST)"] = pd.to_datetime(df["Last Occurred (ST)"], errors="coerce").astype(int) // 10**9
    df["Acknowledged On (ST)"] = pd.to_datetime(df["Acknowledged On (ST)"], errors="coerce").astype(int) // 10**9
    return df

# === Fungsi prediksi severity untuk tanggal target ===
def predict_severity_for_date(model, scaler, reference_df, target_date: str):
    # Ambil 1 baris acak dari data referensi
    if "Severity" in reference_df.columns:
        base_row = reference_df.drop(columns=["Severity"]).sample(n=1).copy()
    else:
        base_row = reference_df.sample(n=1).copy()

    # Set waktu ke tanggal target
    target_unix = int(pd.Timestamp(target_date).timestamp())
    base_row["Last Occurred (ST)"] = target_unix
    base_row["Acknowledged On (ST)"] = target_unix + np.random.randint(60, 3600)

    # Normalisasi dan reshape
    X_scaled = scaler.transform(base_row)
    X_lstm = X_scaled.reshape((1, 1, X_scaled.shape[1]))

    # Prediksi
    y_pred = np.argmax(model.predict(X_lstm), axis=1)[0]
    base_row["Predicted Severity"] = y_pred
    return base_row

# === MAIN ===
# Load model dan scaler
model = load_model("retrained_lstm_alarm_model.h5")
df_train = pd.read_csv("Cleaned_Merged_data_alarm.csv")
df_train["Last Occurred (ST)"] = pd.to_datetime(df_train["Last Occurred (ST)"]).astype(int) // 10**9
df_train["Acknowledged On (ST)"] = pd.to_datetime(df_train["Acknowledged On (ST)"]).astype(int) // 10**9
scaler = StandardScaler()
scaler.fit(df_train.drop(columns=["Severity"]))

# Simulasi input teknisi
uploaded_csv = "dummy_input_april22_25.csv"  # Ganti sesuai file upload
ref_data = process_uploaded_alarm(uploaded_csv)

# Prediksi severity untuk tanggal masa depan (misal 30 April)
predicted = predict_severity_for_date(model, scaler, ref_data, "2025-05-01")
print("\n=== Prediksi Severity untuk 31 April ===")
print(predicted)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step

=== Prediksi Severity untuk 31 April ===
   alarm_description  Alarm ID  Alarm Source  Location Info  \
4                  0     13669            16            361   

   Other Information  Last Occurred (ST)  Acknowledged On (ST)  \
4                196          1746057600            1746060850   

   Fiber/Cable Name  Cleared By  Acknowledged By  Clearance Status  \
4                 1           0                1                 0   

   Acknowledgement Status  Alarm Serial Number  Predicted Severity  
4                       1             25430692                   1  
